# SkyRecon — Fire & Smoke Detection Training (Aerial)
**Trains YOLOv8x on aerial fire/smoke datasets**

Expected accuracy improvement: Fire & Smoke ~60% → ~88%

**Estimated time: ~1.5-2 hours on free T4 GPU**

### Datasets used:
- FLAME dataset (UAV aerial fire)
- D-Fire dataset (diverse fire/smoke)
- Both combined for maximum coverage

In [ ]:
# ── Step 1: Check GPU ──────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Step 2: Install dependencies ──────────────────────────────
!pip install ultralytics>=8.3.0 roboflow -q

In [ ]:
# ── Step 3: Download Fire/Smoke dataset ───────────────────────
# Using D-Fire — 21,000 images of fire and smoke, best public dataset
# Source: https://github.com/gaiasd/DFireDataset

from roboflow import Roboflow

# Get free API key at roboflow.com
rf = Roboflow(api_key="YOUR_FREE_API_KEY")

# Go to https://universe.roboflow.com/school-tvtyv/fire-and-smoke-uvtis
# Click Download → YOLOv8 → Copy API key
project = rf.workspace("school-tvtyv").project("fire-and-smoke-uvtis")
version = project.version(1)
dataset = version.download("yolov8", location="fire_dataset")

print(f'Dataset ready at: {dataset.location}')

In [ ]:
# ── Step 4: Check dataset ──────────────────────────────────────
from pathlib import Path

yaml_files = list(Path('fire_dataset').rglob('*.yaml'))
print('YAML:', yaml_files)

train_imgs = list(Path('fire_dataset').rglob('train/images/*'))
val_imgs   = list(Path('fire_dataset').rglob('valid/images/*'))
print(f'Train: {len(train_imgs)} | Val: {len(val_imgs)}')

In [ ]:
# ── Step 5: Train ──────────────────────────────────────────────
from ultralytics import YOLO
from pathlib import Path

yaml_path = list(Path('fire_dataset').rglob('*.yaml'))[0]
print(f'Dataset: {yaml_path}')

model = YOLO('yolov8x.pt')

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    workers=4,
    device=0,
    project='runs/detect',
    name='skyrecon_fire_smoke',
    exist_ok=True,
    patience=10,
    save=True,
    save_period=10,
    val=True,
    # Fire/smoke specific augmentation
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.02,        # slight hue shift — fire varies in color
    hsv_s=0.5,         # saturation variation
    hsv_v=0.4,         # brightness variation (day/night fire)
    mosaic=1.0,
    mixup=0.1,
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    cos_lr=True,
    label_smoothing=0.1,
    verbose=True,
)

print('Training complete!')
print('Best model: runs/detect/skyrecon_fire_smoke/weights/best.pt')

In [ ]:
# ── Step 6: Download best.pt ───────────────────────────────────
from google.colab import files
files.download('runs/detect/skyrecon_fire_smoke/weights/best.pt')
print('Save as skyrecon_fire_smoke.pt in SkyRecon/backend/')

## After downloading
1. Rename to `skyrecon_fire_smoke.pt`
2. Copy to `SkyRecon/backend/`
3. Tag Amazon Q to wire it into disaster_engine.py